# RAG Assignment: Baseline vs. Re-Ranking
**Document:** Magic: The Gathering Comprehensive Rules

Two retrieval pipelines over the same chunked MTG rulebook:
- **Pipeline 1 (Baseline):** Bi-Encoder → Top-10 → Top-3 → LLM
- **Pipeline 2 (Re-Ranking):** Bi-Encoder Top-10 → Cross-Encoder re-rank → Top-3 → LLM

In [58]:
# Install all required packages into the active venv/kernel
import subprocess, sys

pkgs = [
    "pymupdf",               # PDF parsing
    "sentence-transformers", # Bi-Encoder + Cross-Encoder
    "scikit-learn",          # cosine_similarity
    "numpy", "pandas",
    "google-genai",          # Gemini API
]
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade'] + pkgs,
    check=True
)
print("All packages installed.")

All packages installed.


In [59]:
import os, time, textwrap, pickle, collections
import pymupdf as fitz
from sentence_transformers import SentenceTransformer, CrossEncoder
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import google.genai as genai
import pandas as pd
from IPython.display import display, Markdown

# ── Paths ────────────────────────────────────────────────────────────────────
PDF_PATH        = "MagicCompRules 20260807.pdf"
EMBED_CACHE     = "embeddings_cache.npy"  # pre-computed embeddings saved to disk
CHUNKS_CACHE    = "chunks_cache.pkl"      # corresponding chunk strings

# ── Chunking ─────────────────────────────────────────────────────────────────
CHUNK_SIZE    = 400
CHUNK_OVERLAP = 80

# ── Retrieval ────────────────────────────────────────────────────────────────
TOP_K_RETRIEVE = 10
TOP_K_FINAL    = 3

# ── Models ───────────────────────────────────────────────────────────────────
BI_ENCODER_MODEL    = "all-MiniLM-L6-v2"
CROSS_ENCODER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
GEMINI_MODEL        = "gemini-3.5-flash-lite"

# ── Gemini rate limits (gemini-3.5-flash-lite free tier) ─────────────────────
RATE_LIMIT_RPM = 15   # requests per minute
RATE_LIMIT_RPD = 500  # requests per day

# ── API key ───────────────────────────────────────────────────────────────────
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")
if not GEMINI_API_KEY:
    raise EnvironmentError("Set GEMINI_API_KEY in your environment before running.")

print("Imports and config OK.")

Imports and config OK.


## Setup: Models, Embeddings & LLM

In [60]:
# ── Rate limiter ─────────────────────────────────────────────────────────────
class GeminiRateLimiter:
    """Sliding-window rate limiter for Gemini API (RPM + RPD)."""
    def __init__(self, rpm=15, rpd=500):
        self.rpm = rpm; self.rpd = rpd
        self._window = collections.deque(); self._total = 0
        self._min_gap = 60.0 / rpm
    def wait(self):
        if self._total >= self.rpd:
            raise RuntimeError(f"Daily cap ({self.rpd} RPD) reached.")
        now = time.monotonic()
        while self._window and now - self._window[0] >= 60.0:
            self._window.popleft()
        if len(self._window) >= self.rpm:
            sleep_for = 60.0 - (now - self._window[0]) + 0.2
            if sleep_for > 0:
                print(f"  [RateLimiter] sleeping {sleep_for:.1f} s ...")
                time.sleep(sleep_for)
            now = time.monotonic()
            while self._window and now - self._window[0] >= 60.0:
                self._window.popleft()
        if self._window:
            gap = now - self._window[-1]
            if gap < self._min_gap:
                time.sleep(self._min_gap - gap)
        self._window.append(time.monotonic()); self._total += 1
    @property
    def calls_made(self): return self._total

rate_limiter = GeminiRateLimiter(rpm=RATE_LIMIT_RPM, rpd=RATE_LIMIT_RPD)

# ── Load models (local cache first — avoids silent 30-60s network hang) ──────
print("Loading Bi-Encoder ...")
try:
    bi_encoder = SentenceTransformer(BI_ENCODER_MODEL, local_files_only=True)
except Exception:
    print("  Not cached, downloading ...")
    bi_encoder = SentenceTransformer(BI_ENCODER_MODEL)

print("Loading Cross-Encoder ...")
try:
    cross_encoder = CrossEncoder(CROSS_ENCODER_MODEL, local_files_only=True)
except Exception:
    print("  Not cached, downloading ...")
    cross_encoder = CrossEncoder(CROSS_ENCODER_MODEL)

# ── Gemini client ────────────────────────────────────────────────────────────
client = genai.Client(api_key=GEMINI_API_KEY)

# ── Extract text from PDF ────────────────────────────────────────────────────
doc = fitz.open(PDF_PATH)
raw_text = "\n".join(page.get_text() for page in doc)
doc.close()
print(f"PDF: {len(raw_text):,} chars extracted.")

# ── Chunk text ───────────────────────────────────────────────────────────────
def chunk_text(text, chunk_size=400, overlap=80):
    chunks, start, step = [], 0, chunk_size - overlap
    while start < len(text):
        c = text[start:start + chunk_size].strip()
        if c: chunks.append(c)
        start += step
    return chunks

# ── Load or compute embeddings (cached to disk to avoid re-encoding) ─────────
if os.path.exists(EMBED_CACHE) and os.path.exists(CHUNKS_CACHE):
    # Fast path: load pre-computed embeddings from disk
    chunk_embeddings = np.load(EMBED_CACHE)
    with open(CHUNKS_CACHE, 'rb') as f:
        chunks = pickle.load(f)
    print(f"Loaded {len(chunks):,} chunks + embeddings from disk cache.")
else:
    # Slow path: encode all chunks and save to disk for next run
    chunks = chunk_text(raw_text, CHUNK_SIZE, CHUNK_OVERLAP)
    print(f"Encoding {len(chunks):,} chunks (runs once, then cached) ...")
    # No progress bar — avoids flooding VSCode output and freezing the renderer
    chunk_embeddings = bi_encoder.encode(
        chunks, batch_size=32, show_progress_bar=False, convert_to_numpy=True
    )
    np.save(EMBED_CACHE, chunk_embeddings)
    with open(CHUNKS_CACHE, 'wb') as f:
        pickle.dump(chunks, f)
    print(f"Done. Embeddings saved to {EMBED_CACHE!r} for future runs.")

print(f"Ready: {len(chunks):,} chunks | embedding shape {chunk_embeddings.shape}")

Loading Bi-Encoder ...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading Cross-Encoder ...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


PDF: 968,290 chars extracted.
Loaded 3,026 chunks + embeddings from disk cache.
Ready: 3,026 chunks | embedding shape (3026, 384)


## Pipeline Functions

In [61]:
PROMPT_TEMPLATE = (
    "You are an expert on the Magic: The Gathering Comprehensive Rules.\n"
    "Answer the question using ONLY the provided context passages.\n"
    "If the answer is not in the context, say so. "
    "Be precise and cite rule numbers where possible.\n\n"
    "Context:\n{context}\n\nQuestion: {question}\n\nAnswer:"
)


def baseline_retrieve(query, top_k=TOP_K_RETRIEVE):
    """Bi-Encoder cosine-similarity retrieval. Returns (chunks, scores, elapsed)."""
    t0 = time.time()
    q_emb = bi_encoder.encode([query], convert_to_numpy=True)
    sims  = cosine_similarity(q_emb, chunk_embeddings)[0]
    idx   = np.argsort(sims)[::-1][:top_k]
    return [chunks[i] for i in idx], [float(sims[i]) for i in idx], time.time() - t0


def rerank(query, candidates, top_k=TOP_K_FINAL):
    """Cross-Encoder re-ranking. Returns (chunks, scores, elapsed)."""
    t0 = time.time()
    scores = cross_encoder.predict([(query, c) for c in candidates])
    idx    = np.argsort(scores)[::-1][:top_k]
    return [candidates[i] for i in idx], [float(scores[i]) for i in idx], time.time() - t0


def generate_answer(query, top_chunks):
    """Build RAG prompt and call Gemini (rate-limited). Returns (answer, elapsed)."""
    context = "\n\n".join(f"[{i+1}] {c}" for i, c in enumerate(top_chunks))
    prompt  = PROMPT_TEMPLATE.format(context=context, question=query)
    rate_limiter.wait()
    t0 = time.time()
    resp = client.models.generate_content(model=GEMINI_MODEL, contents=prompt)
    return resp.text, time.time() - t0


def run_both_pipelines(query):
    """Run Baseline and Re-Ranking pipelines, print comparison, return results dict."""
    # Pipeline 1: Baseline
    top10, top10_sc, t_ret    = baseline_retrieve(query)
    ans1, t_llm1              = generate_answer(query, top10[:TOP_K_FINAL])

    # Pipeline 2: Re-Ranking
    top3r, ce_sc, t_rerank    = rerank(query, top10)
    ans2, t_llm2              = generate_answer(query, top3r)

    # Rank-shift table
    all_ce = cross_encoder.predict([(query, c) for c in top10])
    order  = np.argsort(all_ce)[::-1]
    rows   = []
    for bi_r in range(len(top10)):
        ce_r = int(np.where(order == bi_r)[0][0])
        note = ' <-- top-3' if ce_r < TOP_K_FINAL else ''
        rows.append({
            'Bi Rank': f'#{bi_r+1}', 'CE Rank': f'#{ce_r+1}{note}',
            'Bi Score': f'{top10_sc[bi_r]:.4f}', 'CE Score': f'{all_ce[bi_r]:.4f}',
            'Chunk Preview': top10[bi_r][:55].replace('\n', ' '),
        })

    sep = '-' * 70
    print(f'{sep}\nQUERY: {query}\n{sep}')
    print('\nRANK SHIFT TABLE')
    print(pd.DataFrame(rows).to_string(index=False))
    print('\nPIPELINE 1 — Baseline Top-3')
    for i, (c, s) in enumerate(zip(top10[:TOP_K_FINAL], top10_sc[:TOP_K_FINAL])):
        print(f'  #{i+1} [bi={s:.4f}] {c[:90].replace(chr(10)," ")} ...')
    print('\nPIPELINE 2 — Re-Ranked Top-3')
    for i, (c, s) in enumerate(zip(top3r, ce_sc)):
        print(f'  #{i+1} [ce={s:.4f}] {c[:90].replace(chr(10)," ")} ...')
    print('\nANSWER 1 (Baseline):')
    print(textwrap.fill(ans1, 88, initial_indent='  ', subsequent_indent='  '))
    print('\nANSWER 2 (Re-Ranked):')
    print(textwrap.fill(ans2, 88, initial_indent='  ', subsequent_indent='  '))
    print(f'\nLATENCY  retrieval={t_ret:.2f}s  rerank={t_rerank:.2f}s  '
          f'llm1={t_llm1:.2f}s  llm2={t_llm2:.2f}s  '
          f'total_baseline={t_ret+t_llm1:.2f}s  total_reranked={t_ret+t_rerank+t_llm2:.2f}s')
    print(sep)

    return dict(
        query=query,
        t_retrieve=t_ret, t_rerank=t_rerank,
        t_llm_baseline=t_llm1, t_llm_reranked=t_llm2,
        t_baseline_total=t_ret+t_llm1,
        t_rerank_total=t_ret+t_rerank+t_llm2,
        baseline_top3=top10[:TOP_K_FINAL], reranked_top3=top3r,
        answer_baseline=ans1, answer_reranked=ans2,
    )

print("Pipeline functions ready.")

Pipeline functions ready.


## Sanity Check
Verify the pipeline on a simple factual question before the failure-mode analysis.

In [62]:
sanity = run_both_pipelines("How many cards does each player draw at the start of the game?")

----------------------------------------------------------------------
QUERY: How many cards does each player draw at the start of the game?
----------------------------------------------------------------------

RANK SHIFT TABLE
Bi Rank      CE Rank Bi Score CE Score                                           Chunk Preview
     #1 #3 <-- top-3   0.6483   7.6114 layer has set aside their sideboard, their remaining de
     #2           #6   0.5914   4.0243 the player a choice to draw multiple cards, the affecte
     #3 #1 <-- top-3   0.5854   8.9239 wn. Cards can be put into a player’s  hand by other eff
     #4           #7   0.5589   2.7663 on the starting team, in whatever order that  team like
     #5          #10   0.5571  -0.3023 involved in that loop, as well as for   each player wit
     #6           #5   0.5435   4.3407 does the same.    121.2d If more than one player is ins
     #7           #8   0.5416   1.8702 each player on the active team makes any choices requir
     #8 #2

## Failure Mode Analysis


### F1 : Keyword Trap / Distractor

In [63]:
r1 = run_both_pipelines("Can players respond to a mana ability with an instant spell?")

----------------------------------------------------------------------
QUERY: Can players respond to a mana ability with an instant spell?
----------------------------------------------------------------------

RANK SHIFT TABLE
Bi Rank      CE Rank Bi Score CE Score                                           Chunk Preview
     #1 #1 <-- top-3   0.6536   6.5936 llow the normal rules for  activated or triggered abili
     #2           #8   0.6160   2.4322 mana ability, even if it could put mana into a player’s
     #3 #2 <-- top-3   0.5993   4.5324 egardless of what other effects they may generate or wh
     #4           #5   0.5957   3.7930 ny type that land produced.” If a player taps lands for
     #5          #10   0.5831  -1.7630 ability remains a mana ability even if the game state d
     #6           #6   0.5793   3.6927 at target. They then repeat this process for the remain
     #7           #4   0.5746   4.4256 instant or sorcery spell is a spell ability unless it’s
     #8 #3 <

**Scenario:** Lexical overlap on *instant spell* and *mana ability* causes the Bi-Encoder to retrieve generic mana ability definitions and spell restrictions at ranks 1–3 (Rules 605.5b, 605.1a), none of which mention timing or responding. The authoritative rule—Rule 605.3b (*'An activated mana ability doesn’t go on the stack, so it can’t be targeted, countered, or otherwise responded to'*), is buried at **Rank #8**. The Cross-Encoder promotes Rule 605.3b to **Rank #3**, enabling a definitive answer citing rule 605.3b.

### F2 :Exceptions & Corner Cases

In [64]:
r2 = run_both_pipelines("What happens if a blocking creature is removed from combat before damage is assigned?")

----------------------------------------------------------------------
QUERY: What happens if a blocking creature is removed from combat before damage is assigned?
----------------------------------------------------------------------

RANK SHIFT TABLE
Bi Rank      CE Rank Bi Score CE Score                                           Chunk Preview
     #1           #7   0.7287   4.4160 blocked Creature  An attacking creature once no creatur
     #2          #10   0.7106   3.5294 se ends, whichever comes first. A creature  remains blo
     #3           #8   0.6842   4.2989 creature, or becomes a battle. A creature that’s remove
     #4 #3 <-- top-3   0.6826   5.0729 tacking creature with one or more creatures declared as
     #5 #2 <-- top-3   0.6790   5.2451 creature assigns its combat damage to the player, plane
     #6           #9   0.6697   4.2792 that  would have kept that creature from attacking or b
     #7           #5   0.6570   4.7620 ng). It won’t trigger if the  attacking cre

**Scenario:** The general rule is that attacking creatures deal combat damage. The Bi-Encoder retrieves glossary entries and general turn-phase descriptions at ranks 1–3. The specific exception—Rule 510.1c (a blocked creature with no blockers assigns no damage)—is outside Top-3 at **Rank #5**. The Cross-Encoder promotes Rule 510.1c to **Rank #2**.

### F3 : Negation & Constraints

In [65]:
r3 = run_both_pipelines("Can a player cast a creature spell when the stack is not empty?")

----------------------------------------------------------------------
QUERY: Can a player cast a creature spell when the stack is not empty?
----------------------------------------------------------------------

RANK SHIFT TABLE
Bi Rank      CE Rank Bi Score CE Score                                           Chunk Preview
     #1           #5   0.5418   4.1918 of  the spell as it’s cast are applied, then the spell 
     #2 #3 <-- top-3   0.5259   6.2357 de that player from casting an instant spell don’t  aff
     #3           #9   0.5148  -2.1703 remains in  exile. They may cast that card after the cu
     #4 #2 <-- top-3   0.5110   6.4193 on the stack, then the active player gets priority. Pla
     #5 #1 <-- top-3   0.5025   8.3720 ho has priority may cast a creature card from their han
     #6          #10   0.4881  -2.2619 card becomes a spell and is moved to the top of the sta
     #7           #4   0.4813   4.3766 main phase of their turn, the stack is  empty, and they
     #8  

**Scenario:** The Bi-Encoder retrieves general spellcasting resolution rules (Rule 601.3), land-playing mechanics (Rule 305.1), and special foretell rules at ranks 1–3, none of which specify timing constraints for creature spells. The governing negative timing constraint—Rule 302.1 (*'A player who has priority may cast a creature card from their hand during a main phase of their turn when the stack is empty'*), is outside the Top-3 at **Rank #5**. The Cross-Encoder promotes Rule 302.1 directly to **Rank #1**, allowing the LLM to correctly state that creature spells cannot normally be cast when the stack is not empty.

### F4 : Buried Relevant Chunk

In [66]:
r4 = run_both_pipelines("What happens to an Equipment if the creature it is attached to leaves the battlefield?")

----------------------------------------------------------------------
QUERY: What happens to an Equipment if the creature it is attached to leaves the battlefield?
----------------------------------------------------------------------

RANK SHIFT TABLE
Bi Rank      CE Rank Bi Score CE Score                                           Chunk Preview
     #1           #7   0.6726   3.8181 when it resolves. Spells and other  abilities may also 
     #2           #5   0.6022   4.5528 have the subtype “Equipment.” An Equipment can be attac
     #3           #6   0.5752   4.0572 301.5e If an effect attempts to put an Equipment that i
     #4 #3 <-- top-3   0.5408   5.6630 like other artifact spells. Equipment enter the battlef
     #5           #8   0.5359   3.5521 rule 702.151, “Reconfigure”). An Equipment that loses t
     #6 #2 <-- top-3   0.5139   6.0724 er,  that Aura is put into its owner’s graveyard.    70
     #7           #4   0.4940   5.4404 move.    701.3c Attaching an Aura, Equipme

**Scenario:** The Bi-Encoder places general equipment definitions (Rule 301.5), subtype mechanics, and entering-the-battlefield rules at ranks 1–3, none of which describe what happens when the equipped creature leaves the battlefield. The authoritative state-based action and keyword definition—Rule 704.5n (*'If an Equipment or Fortification is attached to an illegal permanent or to a player, it becomes unattached... It remains on the battlefield'*) and Rule 701.3d (*'becoming unattached... includes if the object leaves the zone it was in'*)—are buried at **Rank #6** and **Rank #9**. The Cross-Encoder promotes Rule 701.3d to **Rank #1** and Rule 704.5n to **Rank #2**, providing the exact rule that the Equipment becomes unattached and remains on the battlefield.

### F5 : Keyword Trap 2

In [67]:
r5 = run_both_pipelines("Does damage dealt to a planeswalker reduce that planeswalker's loyalty counters?")

----------------------------------------------------------------------
QUERY: Does damage dealt to a planeswalker reduce that planeswalker's loyalty counters?
----------------------------------------------------------------------

RANK SHIFT TABLE
Bi Rank      CE Rank Bi Score CE Score                                           Chunk Preview
     #1           #4   0.7364   5.4981 to  that planeswalker, further excess damage may be ass
     #2          #10   0.6816   0.3521 planeswalker has a number of loyalty abilities, which a
     #3           #7   0.6699   3.6744 ed, instead remove a stun counter from it.”    122.1e T
     #4 #1 <-- top-3   0.6553   7.7298 hat would be dealt to an opponent be dealt to a planesw
     #5           #6   0.6137   4.7614 y if none of that permanent’s loyalty abilities have  b
     #6 #2 <-- top-3   0.6118   7.0009 at source’s controller to give the  player that many po
     #7           #9   0.6106   1.0183 value for its power, its power is 0. The same is

**Scenario:** Lexical overlap on *damage*, *planeswalker*, and *loyalty* acts as a keyword magnet, causing the Bi-Encoder to retrieve trample excess damage rules, loyalty ability activation timing, and general counter descriptions at ranks 1–3, none of which answer whether damage removes loyalty counters. The authoritative rules—Rule 306.8 (*'Damage dealt to a planeswalker results in that many loyalty counters being removed from it'*) and Rule 120.3c—are buried at **Rank #4** and **Rank #6**. The Cross-Encoder promotes Rule 306.8 directly to **Rank #1** and Rule 120.3c to **Rank #2**, enabling the LLM to give the definitive rule-cited answer.

### F6 : Complex Interaction / Restriction

In [68]:
r6 = run_both_pipelines("Can a creature with the defender ability be declared as an attacking creature?")

----------------------------------------------------------------------
QUERY: Can a creature with the defender ability be declared as an attacking creature?
----------------------------------------------------------------------

RANK SHIFT TABLE
Bi Rank      CE Rank Bi Score CE Score                                           Chunk Preview
     #1 #1 <-- top-3   0.6196   6.4978 an attacking creature refers to a  defending player, or
     #2 #3 <-- top-3   0.5956   5.6911 e attacking  creature’s controller. In the case of a ba
     #3           #5   0.5944   4.5596 of attacking  creatures must be legal as a whole. See r
     #4           #6   0.5763   3.5181 to the battlefield attacking either a player not in the
     #5           #9   0.5660   0.8342 ng that battle with creatures they control.  Creatures 
     #6           #8   0.5587   1.1513 types, only its controller can be its protector.    310
     #7           #7   0.5437   3.2253 ne specific defending player, not to all of the de

**Scenario:**  Lexical overlap with attack declarations pulls general attacking player and combat structure rules into Bi-Encoder ranks 1–3 (Rules 802.2, 508.8, 805.10c). None of those passages mention defender. The governing static restriction—Rule 702.3b (*'A creature with defender can’t attack'*), is buried at **Rank #8**. The Cross-Encoder promotes Rule 702.3 directly to **Rank #2**.

## Results Summary

In [72]:
all_results = [sanity, r1, r2, r3, r4, r5, r6]
labels = [
    "Sanity", "FM1: Keyword Trap (responding to mana ability)", "FM2: Exception (removed blocker damage)",
    "FM3: Negation (creature spell stack empty)", "FM4: Buried chunk (equipment creature leaves)",
    "FM5: Keyword Trap 2 (planeswalker damage loyalty)", "FM6: Restriction (defender attacking)",
]

timing_df = pd.DataFrame([{
    'Query': lbl,
    'Retrieval (s)': f'{r["t_retrieve"]:.3f}',
    'Re-rank (s)':   f'{r["t_rerank"]:.3f}',
    'LLM P1 (s)':    f'{r["t_llm_baseline"]:.3f}',
    'LLM P2 (s)':    f'{r["t_llm_reranked"]:.3f}',
    'Total P1 (s)':  f'{r["t_baseline_total"]:.3f}',
    'Total P2 (s)':  f'{r["t_rerank_total"]:.3f}',
} for lbl, r in zip(labels, all_results)])

display(timing_df)

avg_p1 = sum(r['t_baseline_total'] for r in all_results) / len(all_results)
avg_p2 = sum(r['t_rerank_total']   for r in all_results) / len(all_results)
print(f'Avg total Pipeline 1: {avg_p1:.2f}s | Pipeline 2: {avg_p2:.2f}s')
print(f'Total LLM calls made: {rate_limiter.calls_made}')

# Side-by-side qualitative comparison for failure modes
print('\n' + '='*70)
for lbl, r in zip(labels[1:], all_results[1:]):
    display(Markdown(f'### {lbl}'))
    display(Markdown(f'**Query:** {r["query"]}'))
    display(Markdown('**Answer 1 (Baseline):**\n' + r['answer_baseline']))
    display(Markdown('**Answer 2 (Re-Ranked):**\n' + r['answer_reranked']))

,Query,Retrieval (s),Re-rank (s),LLM P1 (s),LLM P2 (s),Total P1 (s),Total P2 (s)
0,Sanity,0.026,0.111,1.052,0.695,1.078,0.832
1,FM1: Keyword Trap (responding to mana ability),0.024,0.101,0.993,0.885,1.017,1.011
2,FM2: Exception (removed blocker damage),0.044,0.101,0.842,0.986,0.885,1.130
3,FM3: Negation (creature spell stack empty),0.038,0.108,1.133,0.941,1.171,1.087
4,FM4: Buried chunk (equipment creature leaves),0.024,0.124,2.068,0.898,2.092,1.046
5,FM5: Keyword Trap 2 (planeswalker damage loyalty),0.021,0.109,1.016,0.776,1.037,0.906
6,FM6: Restriction (defender attacking),0.027,0.098,5.856,1.039,5.882,1.164


Avg total Pipeline 1: 1.88s | Pipeline 2: 1.03s
Total LLM calls made: 14



### FM1: Keyword Trap (responding to mana ability)

**Query:** Can players respond to a mana ability with an instant spell?

**Answer 1 (Baseline):**
Based on the provided context, there is no mention of whether players can respond to a mana ability with an instant spell. Therefore, the answer is not in the context.

...

**Answer 2 (Re-Ranked):**
Yes. Damage dealt to a planeswalker results in that many loyalty counters being removed from it (rules 306.8 and 120.3c).

### FM6: Restriction (defender attacking)

**Query:** Can a creature with the defender ability be declared as an attacking creature?

**Answer 1 (Baseline):**
Based on the provided context, there is no mention of whether a creature with the defender ability can be declared as an attacking creature. Therefore, the answer is not in the context.

**Answer 2 (Re-Ranked):**
Based on the provided context, no. A creature with defender cannot attack (Rule 702.3b).